In [ ]:
%pip install -q ultralytics opencv-python matplotlib pandas pyyaml


In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import yaml
from ultralytics import YOLO

print("Libraries imported successfully.")


In [ ]:
CLASS_NAMES = {
    0: "Ready",
    1: "Broken",
    2: "Bent"
}

print("YOLO classes:")
for k, v in CLASS_NAMES.items():
    print(f"{k}: {v}")


In [ ]:
dataset_path = "Agarbatti_Dataset"

data_config = {
    "path": dataset_path,
    "train": "train/images",
    "val": "val/images",
    "names": CLASS_NAMES
}

with open("data.yaml", "w") as f:
    yaml.dump(data_config, f, sort_keys=False)

print(open("data.yaml").read())


In [ ]:
print("Training cell ready.")


In [ ]:
IMAGE_PATH = "agarbatti_image.jpg"

print("Image:", IMAGE_PATH)
print("Exists:", os.path.exists(IMAGE_PATH))


In [ ]:
def capture_agarbatti_image(output_file="agarbatti_image.jpg", camera_index=0):
    cap = cv2.VideoCapture(camera_index)

    if not cap.isOpened():
        print("ERROR: Camera could not be opened.")
        return None

    print("Press SPACE to capture; ESC to exit.")

    while True:
        ret, frame = cap.read()
        if not ret:
            print("ERROR: Cannot read camera frame.")
            break

        cv2.imshow("Smart Agarbatti Inspection", frame)
        key = cv2.waitKey(1) & 0xFF

        if key == 32:
            cv2.imwrite(output_file, frame)
            cap.release()
            cv2.destroyAllWindows()
            print("Captured:", output_file)
            return output_file

        if key == 27:
            break

    cap.release()
    cv2.destroyAllWindows()
    return None


In [ ]:
def preprocess_for_opencv(image):
    if image is None:
        raise ValueError("Input image is empty.")

    denoised = cv2.GaussianBlur(image, (3, 3), 0)

    lab = cv2.cvtColor(denoised, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)

    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    enhanced_l = clahe.apply(l)

    enhanced = cv2.merge([enhanced_l, a, b])
    return cv2.cvtColor(enhanced, cv2.COLOR_LAB2BGR)

def load_and_preprocess(image_path):
    image = cv2.imread(image_path)
    if image is None:
        raise FileNotFoundError(f"Cannot read: {image_path}")
    return image, preprocess_for_opencv(image)

if os.path.exists(IMAGE_PATH):
    original, processed = load_and_preprocess(IMAGE_PATH)

    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.imshow(cv2.cvtColor(original, cv2.COLOR_BGR2RGB))
    plt.title("Original")
    plt.axis("off")

    plt.subplot(1, 2, 2)
    plt.imshow(cv2.cvtColor(processed, cv2.COLOR_BGR2RGB))
    plt.title("OpenCV enhanced")
    plt.axis("off")
    plt.show()


In [ ]:
MODEL_PATH = "agarbatti_runs/quality_grading/weights/best.pt"

if os.path.exists(MODEL_PATH):
    model = YOLO(MODEL_PATH)
    print("Custom YOLOv8 model loaded.")
else:
    model = None
    print("Custom model not found. Train the model first or update MODEL_PATH.")


In [ ]:
def run_yolo_detection(model, image_path, confidence=0.35):
    if model is None:
        raise ValueError("YOLO model is not loaded.")
    return model(image_path, conf=confidence, verbose=False)


In [ ]:
def measure_detected_object(box):
    x1, y1, x2, y2 = map(int, box)

    width_px = max(0, x2 - x1)
    height_px = max(0, y2 - y1)

    long_side = max(width_px, height_px)
    short_side = max(1, min(width_px, height_px))

    return {
        "width_px": width_px,
        "height_px": height_px,
        "aspect_ratio": round(long_side / short_side, 2)
    }


In [ ]:
PIXELS_PER_MM = None

def add_mm_measurements(features, pixels_per_mm=PIXELS_PER_MM):
    out = dict(features)

    if pixels_per_mm is not None and pixels_per_mm > 0:
        out["width_mm"] = round(out["width_px"] / pixels_per_mm, 2)
        out["height_mm"] = round(out["height_px"] / pixels_per_mm, 2)
    else:
        out["width_mm"] = None
        out["height_mm"] = None

    return out


In [ ]:
def make_foreground_mask(crop):
    gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
    gray = cv2.GaussianBlur(gray, (5, 5), 0)

    _, mask = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)

    kernel = np.ones((3, 3), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)

    return mask

def extract_colour_features(crop):
    mask = make_foreground_mask(crop)

    if cv2.countNonZero(mask) < 20:

        mask = np.ones(mask.shape, dtype=np.uint8) * 255

    hsv = cv2.cvtColor(crop, cv2.COLOR_BGR2HSV)

    mean_bgr = cv2.mean(crop, mask=mask)[:3]
    mean_hsv = cv2.mean(hsv, mask=mask)[:3]

    gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
    brightness = cv2.mean(gray, mask=mask)[0]

    return {
        "mean_blue": round(mean_bgr[0], 2),
        "mean_green": round(mean_bgr[1], 2),
        "mean_red": round(mean_bgr[2], 2),
        "mean_hue": round(mean_hsv[0], 2),
        "mean_saturation": round(mean_hsv[1], 2),
        "mean_value": round(mean_hsv[2], 2),
        "brightness": round(brightness, 2)
    }


In [ ]:
def extract_all_features(image_path, results):
    image = cv2.imread(image_path)
    if image is None:
        raise FileNotFoundError(image_path)

    feature_records = []

    for result in results:
        names = result.names

        for box_obj in result.boxes:
            x1, y1, x2, y2 = map(int, box_obj.xyxy[0].tolist())

            confidence = float(box_obj.conf[0])
            class_id = int(box_obj.cls[0])
            class_name = names[class_id]

            x1 = max(0, x1)
            y1 = max(0, y1)
            x2 = min(image.shape[1], x2)
            y2 = min(image.shape[0], y2)

            crop = image[y1:y2, x1:x2]
            if crop.size == 0:
                continue

            size_features = measure_detected_object([x1, y1, x2, y2])
            size_features = add_mm_measurements(size_features)

            colour_features = extract_colour_features(crop)

            feature_records.append({
                "class": class_name,
                "confidence": round(confidence * 100, 2),
                "box": [x1, y1, x2, y2],
                **size_features,
                **colour_features
            })

    return feature_records


In [ ]:
DRYING_STATUS = "READY_TO_INSPECT"

def validate_drying_status(status):
    allowed = {"READY_TO_INSPECT", "UNDER_DRYING", "UNKNOWN"}
    if status not in allowed:
        raise ValueError(f"DRYING_STATUS must be one of {allowed}")
    return status


In [ ]:
MIN_WIDTH_PX = None
MAX_WIDTH_PX = None
MIN_HEIGHT_PX = None
MAX_HEIGHT_PX = None

MIN_BRIGHTNESS = None
MAX_BRIGHTNESS = None

def within_limits(value, low=None, high=None):
    if value is None:
        return True
    if low is not None and value < low:
        return False
    if high is not None and value > high:
        return False
    return True

def quality_decision(record, drying_status=DRYING_STATUS, min_confidence=50):
    validate_drying_status(drying_status)

    cls = record["class"]
    conf = record["confidence"]
    reasons = []

    if drying_status == "UNDER_DRYING":
        return {
            "status": "UNDER-DRYING",
            "grade": "GRADE C",
            "action": "KEEP DRYING",
            "reason": "ESP32 drying status says the batch is not ready for inspection."
        }

    if drying_status == "UNKNOWN":
        return {
            "status": "UNKNOWN DRYING STATUS",
            "grade": "MANUAL CHECK",
            "action": "CHECK SENSORS",
            "reason": "Drying status is unavailable or invalid."
        }

    if cls == "Broken":
        return {
            "status": "BROKEN",
            "grade": "REJECT",
            "action": "REMOVE",
            "reason": "YOLOv8 detected a broken agarbatti."
        }

    if cls == "Bent":
        return {
            "status": "BENT",
            "grade": "REJECT",
            "action": "REMOVE",
            "reason": "YOLOv8 detected a bent agarbatti."
        }

    if conf < min_confidence:
        return {
            "status": "LOW CONFIDENCE",
            "grade": "MANUAL CHECK",
            "action": "RECHECK IMAGE",
            "reason": f"YOLO confidence is only {conf:.1f}%."
        }

    if not within_limits(record["width_px"], MIN_WIDTH_PX, MAX_WIDTH_PX):
        reasons.append("width outside calibrated range")

    if not within_limits(record["height_px"], MIN_HEIGHT_PX, MAX_HEIGHT_PX):
        reasons.append("length/height outside calibrated range")

    if not within_limits(record["brightness"], MIN_BRIGHTNESS, MAX_BRIGHTNESS):
        reasons.append("brightness outside calibrated range")

    if reasons:
        return {
            "status": "READY BUT OUT OF SPEC",
            "grade": "GRADE C",
            "action": "DOWNGRADE / CHECK",
            "reason": "; ".join(reasons)
        }

    return {
        "status": "READY",
        "grade": "GRADE A",
        "action": "ACCEPT",
        "reason": "YOLO found Ready, drying is complete, and enabled OpenCV checks passed."
    }


In [ ]:
def complete_agarbatti_pipeline(image_path, model, drying_status=DRYING_STATUS):
    if model is None:
        raise ValueError("YOLO model is not loaded.")

    print("=" * 65)
    print("SMART AGARBATTI QUALITY INSPECTION")
    print("=" * 65)

    results = run_yolo_detection(model, image_path)
    print("✓ YOLOv8 detection completed")

    records = extract_all_features(image_path, results)
    print(f"✓ OpenCV features extracted for {len(records)} object(s)")

    final_records = []

    for i, record in enumerate(records, start=1):
        decision = quality_decision(record, drying_status=drying_status)

        final_records.append({
            "Agarbatti_No": i,
            **record,
            **decision
        })

    return results, final_records


In [ ]:
if model is not None and os.path.exists(IMAGE_PATH):
    results, final_records = complete_agarbatti_pipeline(
        IMAGE_PATH, model, DRYING_STATUS
    )

    df = pd.DataFrame(final_records)
    display(df)
else:
    print("Provide IMAGE_PATH and a trained MODEL_PATH first.")


In [ ]:
if model is not None and os.path.exists(IMAGE_PATH):
    results = run_yolo_detection(model, IMAGE_PATH)
    annotated = results[0].plot()

    plt.figure(figsize=(12, 8))
    plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
    plt.title("YOLOv8 Agarbatti Detection")
    plt.axis("off")
    plt.show()


In [ ]:
def batch_summary(final_records):
    summary = {
        "total": len(final_records),
        "grade_a": 0,
        "grade_c": 0,
        "reject": 0,
        "manual_check": 0
    }

    for r in final_records:
        grade = r.get("grade", "")
        if grade == "GRADE A":
            summary["grade_a"] += 1
        elif grade == "GRADE C":
            summary["grade_c"] += 1
        elif grade == "REJECT":
            summary["reject"] += 1
        else:
            summary["manual_check"] += 1

    return summary

if "final_records" in globals():
    summary = batch_summary(final_records)

    print("\nBATCH QUALITY SUMMARY")
    print("=" * 35)
    print("Total inspected:", summary["total"])
    print("GRADE A:", summary["grade_a"])
    print("GRADE C:", summary["grade_c"])
    print("REJECT:", summary["reject"])
    print("MANUAL CHECK:", summary["manual_check"])
